# Iris Species Classification

Compare Logistic Regression and a depth-limited Decision Tree on four flower measurements. This educational Datheon task demonstrates data exploration, leakage-aware pipelines, classification metrics, and visual interpretation. Results are from a small benchmark, not a deployed system.

**Run:** In Colab select **Runtime → Run all**. Locally follow the [project README](README.md). The runner uses the same notebook cells. Random seeds are fixed at 42.

<a href="https://colab.research.google.com/github/primrosemandy93-maker/Datheon_Machine_Learning_-_Data_Science_Internship/blob/main/task_1_iris/Task_1_Iris_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Tools and reproducible output paths

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
)

# Supports execution from the repository root or this project folder.
PROJECT_DIR = next((p / "task_1_iris" for p in [Path.cwd(), *Path.cwd().parents] if (p / "task_1_iris").is_dir()), Path.cwd())
OUTPUT_DIR = PROJECT_DIR / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Consistent presentation across the portfolio.
sns.set_theme(style="whitegrid", palette=["#2563EB", "#0D9488", "#64748B"], font_scale=1.05)
plt.rcParams.update({"axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False, "figure.facecolor": "white", "axes.labelcolor": "#334155", "text.color": "#0F172A"})


## 2. Dataset and quality checks

In [2]:
# Load the Iris flower dataset.
iris = load_iris(as_frame=True)

# X contains the flower measurements.
X = iris.data

# y contains the flower species as numbers: 0, 1, or 2.
y = iris.target

# Create a table with readable species names.
df = X.copy()
df["species"] = y.map(dict(enumerate(iris.target_names)))

display(df.head())

print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())

Dataset shape: (150, 5)

Missing values:
sepal length (cm)    0
sepal width (cm)     0
petal length (cm)    0
petal width (cm)     0
species              0
dtype: int64


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


## 3. Explore the measurements

In [3]:
# Compare flower measurements across the three species.
sns.pairplot(
    df,
    hue="species",
    diag_kind="hist",
    corner=True,
)

plt.suptitle("Iris flower measurements by species", y=1.02)
plt.savefig(OUTPUT_DIR / "feature_relationships.png", dpi=140, bbox_inches="tight")
plt.show()

![Saved figure](results/feature_relationships.png)

## 4. Hold out a stratified test set

In [4]:
# Use 80% of the flowers for training and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training flowers:", len(X_train))
print("Testing flowers:", len(X_test))

Training flowers: 120
Testing flowers: 30


## 5. Fit preprocessing and models on training data

In [5]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42,
        )),
    ]),
    "Decision Tree": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", DecisionTreeClassifier(
            max_depth=3,
            random_state=42,
        )),
    ]),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name}: training complete")

Logistic Regression: training complete
Decision Tree: training complete


## 6. Evaluate both models on the held-out flowers

In [6]:
results = []

for name, model in models.items():
    predictions = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, average="macro", zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, average="macro", zero_division=0
        ),
        "F1-score": f1_score(
            y_test, predictions, average="macro", zero_division=0
        ),
    })

    print(f"\n{name}")
    print(classification_report(
        y_test,
        predictions,
        labels=[0, 1, 2],
        target_names=iris.target_names,
        zero_division=0,
    ))

results_df = pd.DataFrame(results)
display(results_df.round(3))


Logistic Regression
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30


Decision Tree
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



,Model,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.933,0.933,0.933,0.933
1,Decision Tree,0.967,0.970,0.967,0.967


## 7. Inspect classification errors

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(
        model,
        X_test,
        y_test,
        display_labels=iris.target_names,
        cmap="Blues",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(name)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrices.png", dpi=140, bbox_inches="tight")
plt.show()

![Saved figure](results/confusion_matrices.png)

## 8. Visualise separate two-feature models

In [8]:
# Use two measurements so we can draw a flat graph.
features = ["petal length (cm)", "petal width (cm)"]

X_train_2d = X_train[features]
X_test_2d = X_test[features]

# Create a grid covering the training measurements.
x_min, x_max = X_train_2d.iloc[:, 0].min() - 0.5, X_train_2d.iloc[:, 0].max() + 0.5
y_min, y_max = X_train_2d.iloc[:, 1].min() - 0.5, X_train_2d.iloc[:, 1].max() + 0.5

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 250),
    np.linspace(y_min, y_max, 250),
)

grid = pd.DataFrame(
    np.c_[xx.ravel(), yy.ravel()],
    columns=features,
)

visual_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=3, random_state=42
    ),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, classifier) in zip(axes, visual_models.items()):
    model_2d = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", classifier),
    ])

    model_2d.fit(X_train_2d, y_train)
    predictions = model_2d.predict(grid).reshape(xx.shape)

    ax.contourf(
        xx, yy, predictions,
        levels=[-0.5, 0.5, 1.5, 2.5],
        cmap="viridis",
        alpha=0.25,
    )

    for label, species in enumerate(iris.target_names):
        mask = y_test == label

        ax.scatter(
            X_test_2d.loc[mask, features[0]],
            X_test_2d.loc[mask, features[1]],
            label=species,
            edgecolors="black",
        )

    ax.set_xlabel(features[0])
    ax.set_ylabel(features[1])
    ax.set_title(name)
    ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "decision_boundaries.png", dpi=140, bbox_inches="tight")
plt.show()

![Saved figure](results/decision_boundaries.png)

## 9. Export measured results

In [9]:
print(results_df.round(3).to_string(index=False))
results_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)


              Model  Accuracy  Precision  Recall  F1-score
Logistic Regression     0.933      0.933   0.933     0.933
      Decision Tree     0.967      0.970   0.967     0.967


## 10. Robustness: repeated training-only validation and a simple baseline

In [10]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate

# Keep the 30-flower test set out of this robustness check.
repeated_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
cv_models = {"Majority-class baseline": DummyClassifier(strategy="most_frequent"), **models}
cv_rows = []
cv_fold_rows = []
for name, estimator in cv_models.items():
    scores = cross_validate(estimator, X_train, y_train, cv=repeated_cv,
                            scoring={"accuracy": "accuracy", "macro_f1": "f1_macro"},
                            n_jobs=1, error_score="raise")
    cv_rows.append({"Model": name,
                    "Mean accuracy": scores["test_accuracy"].mean(),
                    "Accuracy fold SD": scores["test_accuracy"].std(),
                    "Mean macro F1": scores["test_macro_f1"].mean(),
                    "Macro F1 fold SD": scores["test_macro_f1"].std()})
    for fold, (acc, f1) in enumerate(zip(scores["test_accuracy"], scores["test_macro_f1"]), 1):
        cv_fold_rows.append({"Model": name, "Split": fold, "Accuracy": acc, "Macro F1": f1})
repeated_results = pd.DataFrame(cv_rows)
repeated_results.to_csv(OUTPUT_DIR / "repeated_validation.csv", index=False)
pd.DataFrame(cv_fold_rows).to_csv(OUTPUT_DIR / "repeated_validation_folds.csv", index=False)
display(repeated_results.round(4))
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(repeated_results["Model"], repeated_results["Mean accuracy"],
        xerr=repeated_results["Accuracy fold SD"], color=["#94a3b8", "#2563eb", "#0d9488"], capsize=4)
ax.set_xlim(0, 1)
ax.set_xlabel("Mean validation accuracy (error bars: fold standard deviation)")
ax.set_title("Iris: 5-fold stratified validation repeated 10 times on training data")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "repeated_validation.png", dpi=140, bbox_inches="tight")
plt.show()
print("50 split scores per model; folds overlap, so SD is not a confidence interval.")
print("These results assess stability on the same small benchmark, not external validity.")


50 split scores per model; folds overlap, so SD is not a confidence interval.
These results assess stability on the same small benchmark, not external validity.


,Model,Mean accuracy,Accuracy fold SD,Mean macro F1,Macro F1 fold SD
0,Majority-class baseline,0.3333,0.0000,0.1667,0.0000
1,Logistic Regression,0.9600,0.0322,0.9597,0.0326
2,Decision Tree,0.9508,0.0462,0.9502,0.0471


![Saved figure](results/repeated_validation.png)

## Results and Conclusion

Both models were trained on 120 Iris flowers and evaluated on 30
unseen flowers using an 80/20 stratified split.

| Model | Accuracy | Macro Precision | Macro Recall | Macro F1 |
|---|---:|---:|---:|---:|
| Logistic Regression | 0.933 | 0.933 | 0.933 | 0.933 |
| Decision Tree | 0.967 | 0.970 | 0.967 | 0.967 |

The Decision Tree achieved the higher test accuracy, correctly
classifying 29 out of 30 flowers. Logistic Regression correctly
classified 28 out of 30 flowers.

Feature scaling was fitted within each pipeline using only the
training data to prevent data leakage.

The decision boundary plots used separate models trained on two
petal measurements for visualization. The evaluation table above
uses models trained on all four measurements.

These results come from one small test set. A difference of one
correct prediction does not establish that the Decision Tree will
always perform better. The additional training-only repeated stratified validation compares both models with a majority-class baseline. Fold variation should be interpreted as descriptive stability, not independent evidence or a confidence interval.